In [ ]:
LOG_PREFIX = "PLEASE_CHANGE!"  # Change this first.

from pathlib import Path
import re
import math
import numpy as np
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
_roots = [_cwd, *_cwd.parents]
REPO_ROOT = next((root for root in _roots if (root / "scripts" / "ae").is_dir() and (root / "log").is_dir()), _cwd)
LOG_DIR = REPO_ROOT / "log" / LOG_PREFIX
FIGURE_DIR = REPO_ROOT / "figures" / LOG_PREFIX
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
N_QUERIES = 10000
GUSTANN_REPEAT = 500  # From -R 500 in run_sift100m_gustann.sh.
PAGE_BYTES = 4096
GIB = 1024 ** 3
BANDWIDTH_YLIM = (0, 50)
COLORS = {"GustANN": "#0072B2", "BANG": "#D55E00", "DiskANN": "#009E73", "Starling": "#CC79A7", "SPANN": "#E69F00", "RUMMY": "#56B4E9"}
MARKERS = {"GustANN": "o", "BANG": "s", "DiskANN": "^", "Starling": "D", "SPANN": "P", "RUMMY": "X"}

def read_log(system):
    path = LOG_DIR / f"log_sift100m_{system.lower()}.log"
    return path.read_text(errors="replace") if path.exists() else ""

def point(system, k, ef, qps, recall, pages, pages_per_query=True):
    if pages is None or qps is None:
        bandwidth = math.nan
    elif pages_per_query:
        bandwidth = pages * PAGE_BYTES * qps / GIB
    else:
        # GustANN reports total pages for all N_QUERIES * R requests.
        bandwidth = pages * PAGE_BYTES * qps / (N_QUERIES * GUSTANN_REPEAT * GIB)
    return {"system": system, "k": k, "ef": ef, "qps": qps, "recall": recall, "pages": pages, "bandwidth": bandwidth}

def pcie_rx_average(text, threshold=1024):
    match = re.search(r"=== PCIe LOG ===(.*?)={5,}", text, re.S)
    if not match:
        raise ValueError("PCIe monitor section is missing")
    runs, active = [], []
    for line in match.group(1).splitlines():
        sample = re.fullmatch(r"\s*(\d+)\s+(\d+)\s+(\d+)\s*", line)
        if not sample:
            continue
        rx = int(sample.group(2))
        if rx > threshold:
            active.append(rx)
        elif active:
            runs.append(active)
            active = []
    if active:
        runs.append(active)
    if not runs or len(runs[-1]) < 3:
        raise ValueError("Not enough PCIe search samples after excluding boundary seconds")
    samples = runs[-1][1:-1]
    return float(np.mean(samples)) / 1024, samples

print(f"Reading {LOG_DIR}")
if not LOG_DIR.is_dir():
    raise FileNotFoundError(f"Log directory does not exist: {LOG_DIR}")


In [ ]:
def parse_gustann():
    text = read_log("gustann")
    rows, current, time_s, pages = [], None, None, None
    for line in text.splitlines():
        m = re.search(r"\[REPORT\] TOP (\d+) EF (\d+)", line)
        if m:
            current, time_s, pages = (int(m.group(1)), int(m.group(2))), None, None
            continue
        m = re.search(r"\[REPORT\] Time ([0-9.eE+-]+)", line)
        if m and current:
            time_s = float(m.group(1))
            continue
        m = re.search(r"\[REPORT\] IO ([0-9.eE+-]+)", line)
        if m and current:
            pages = float(m.group(1))
            continue
        m = re.search(r"\[REPORT\] RECALL: ([0-9.eE+-]+)", line)
        if m and current and time_s is not None and pages is not None:
            qps = N_QUERIES * GUSTANN_REPEAT / time_s
            rows.append(point("GustANN", current[0], current[1], qps, float(m.group(1)), pages, pages_per_query=False))
            current = None
    return rows

def parse_bang():
    text, rows, k, ef = read_log("bang"), [], None, None
    for line in text.splitlines():
        m = re.search(r"\[REPORT\] K (\d+)", line)
        if m:
            k = int(m.group(1))
        m = re.search(r"\[REPORT\] EF (\d+)", line)
        if m:
            ef = int(m.group(1))
        m = re.search(r"\[REPORT\] QPS ([0-9.eE+-]+) Recall ([0-9.eE+-]+)", line)
        if m and k is not None and ef is not None:
            # BANG does not report SSD page counts, so its bandwidth is intentionally NaN.
            rows.append(point("BANG", k, ef, float(m.group(1)), float(m.group(2)) / 100.0, None))
    return rows

def parse_spann():
    text, rows, current, qps, pages, recall = read_log("spann"), [], None, None, None, None
    for line in text.splitlines():
        m = re.search(r"\[REPORT\] K (\d+) EF (\d+)", line)
        if m:
            current, qps, pages, recall = (int(m.group(1)), int(m.group(2))), None, None, None
        m = re.search(r"\[REPORT\] QPS ([0-9.eE+-]+)", line)
        if m and current:
            qps = float(m.group(1))
        m = re.search(r"\[REPORT\] IO ([0-9.eE+-]+)", line)
        if m and current:
            pages = float(m.group(1))
        m = re.search(r"\[REPORT\] Recall: ([0-9.eE+-]+)", line)
        if m and current:
            recall = float(m.group(1))
        if current and qps is not None and pages is not None and recall is not None:
            rows.append(point("SPANN", current[0], current[1], qps, recall, pages, pages_per_query=True))
            current = None
    return rows

def parse_disk_table(system):
    text = read_log(system.lower())
    rows_by_k, current_k = {1: [], 10: []}, None
    row_re = re.compile(r"^\s*(\d+)\s+4\s+([0-9.eE+-]+)\s+[0-9.eE+-]+\s+[0-9.eE+-]+\s+([0-9.eE+-]+)\s+")
    for line in text.splitlines():
        m = re.search(r"\[REPORT\] TOP (\d+)", line)
        if m:
            current_k = int(m.group(1))
            continue
        m = row_re.match(line)
        if m and current_k in rows_by_k:
            rows_by_k[current_k].append((int(m.group(1)), float(m.group(2)), float(m.group(3))))
    recall = {1: {}, 10: {}}
    for line in text.splitlines():
        m = re.search(r"sift100m-(\d+)-[^_]+_(\d+)_idx_uint32\.binRecall @ \d+: ([0-9.eE+-]+)", line)
        if m:
            recall[int(m.group(1))][int(m.group(2))] = float(m.group(3))
    out = []
    for k in (1, 10):
        wanted = recall[k]
        table = rows_by_k[k][-len(wanted):] if wanted else []
        for ef, qps, pages in table:
            if ef in wanted:
                out.append(point(system, k, ef, qps, wanted[ef], pages, pages_per_query=True))
    return out

data = parse_gustann() + parse_bang() + parse_disk_table("DiskANN") + parse_disk_table("Starling") + parse_spann()
if not any(row["system"] == "SPANN" for row in data):
    print("ERROR: SPANN has no valid SIFT100M data in this log; all plotted SPANN values are missing.")
if not data:
    raise RuntimeError("No plot data found")

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex="col")
for ax, k in zip(axes[0], (1, 10)):
    for system in COLORS:
        rows = [r for r in data if r["system"] == system and r["k"] == k and np.isfinite(r["qps"]) and np.isfinite(r["recall"]) ]
        if rows:
            ax.plot([r["recall"] for r in rows], [r["qps"] for r in rows], marker=MARKERS[system], color=COLORS[system], label=system, linewidth=1.5, markersize=5)
    ax.set_yscale("log")
    ax.set_ylim(1e2, 1e6)
    ax.set_xlim(0.8, 1.0)
    ax.grid(True, which="both", alpha=0.25)
    ax.set_ylabel("Throughput (QPS)")
for ax, k in zip(axes[1], (1, 10)):
    for system in COLORS:
        rows = [r for r in data if r["system"] == system and r["k"] == k and np.isfinite(r["bandwidth"]) and np.isfinite(r["recall"]) ]
        if rows:
            ax.plot([r["recall"] for r in rows], [r["bandwidth"] for r in rows], marker=MARKERS[system], color=COLORS[system], label=system, linewidth=1.5, markersize=5)
    ax.set_xlim(0.8, 1.0)
    ax.set_ylim(*BANDWIDTH_YLIM)
    ax.set_xlabel("Recall@%d" % k)
    ax.set_ylabel("SSD Bandwidth (GiB/s)")
    ax.grid(True, alpha=0.25)
for ax, k in zip(axes[0], (1, 10)):
    ax.set_xlabel("Recall@%d" % k)
axes[0, 0].set_title("Recall@1 (k=1)")
axes[0, 1].set_title("Recall@10 (k=10)")
fig.suptitle("Figure 7", fontsize=16)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5, frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 0.96))
output = FIGURE_DIR / "figure7_sift100m.pdf"
fig.savefig(output, dpi=300, bbox_inches="tight")
print(f"Saved {output}")
plt.show()


In [ ]:
def read_dataset_log(dataset, system):
    path = LOG_DIR / f"log_{dataset}_{system.lower()}.log"
    return path.read_text(errors="replace") if path.exists() else ""

def parse_gustann_dataset(dataset):
    text = read_dataset_log(dataset, "gustann")
    rows, current, time_s, pages = [], None, None, None
    for line in text.splitlines():
        m = re.search(r"\[REPORT\] TOP (\d+) EF (\d+)", line)
        if m:
            current, time_s, pages = (int(m.group(1)), int(m.group(2))), None, None
        m = re.search(r"\[REPORT\] Time ([0-9.eE+-]+)", line)
        if m and current:
            time_s = float(m.group(1))
        m = re.search(r"\[REPORT\] IO ([0-9.eE+-]+)", line)
        if m and current:
            pages = float(m.group(1))
        m = re.search(r"\[REPORT\] RECALL: ([0-9.eE+-]+)", line)
        if m and current and time_s is not None and pages is not None:
            qps = N_QUERIES * GUSTANN_REPEAT / time_s
            rows.append(point("GustANN", current[0], current[1], qps, float(m.group(1)), pages, pages_per_query=False))
            current = None
    return rows

def parse_bang_dataset(dataset):
    text, rows, k, ef = read_dataset_log(dataset, "bang"), [], None, None
    for line in text.splitlines():
        m = re.search(r"\[REPORT\] K (\d+)", line)
        if m:
            k = int(m.group(1))
        m = re.search(r"\[REPORT\] EF (\d+)", line)
        if m:
            ef = int(m.group(1))
        m = re.search(r"\[REPORT\] QPS ([0-9.eE+-]+) Recall ([0-9.eE+-]+)", line)
        if m and k is not None and ef is not None:
            rows.append(point("BANG", k, ef, float(m.group(1)), float(m.group(2)) / 100.0, None))
    return rows

def parse_spann_dataset(dataset):
    text, rows, current, qps, pages, recall = read_dataset_log(dataset, "spann"), [], None, None, None, None
    for line in text.splitlines():
        m = re.search(r"\[REPORT\] K (\d+) EF (\d+)", line)
        if m:
            current, qps, pages, recall = (int(m.group(1)), int(m.group(2))), None, None, None
        m = re.search(r"\[REPORT\] QPS ([0-9.eE+-]+)", line)
        if m and current:
            qps = float(m.group(1))
        m = re.search(r"\[REPORT\] IO ([0-9.eE+-]+)", line)
        if m and current:
            pages = float(m.group(1))
        m = re.search(r"\[REPORT\] Recall: ([0-9.eE+-]+)", line)
        if m and current:
            recall = float(m.group(1))
        if current and qps is not None and pages is not None and recall is not None:
            rows.append(point("SPANN", current[0], current[1], qps, recall, pages, pages_per_query=True))
            current = None
    return rows

def parse_table_dataset(dataset, system):
    text = read_dataset_log(dataset, system)
    rows_by_k, current_k = {1: [], 10: []}, None
    row_re = re.compile(r"^\s*(\d+)\s+4\s+([0-9.eE+-]+)\s+[0-9.eE+-]+\s+[0-9.eE+-]+\s+([0-9.eE+-]+)\s+")
    for line in text.splitlines():
        m = re.search(r"\[REPORT\] TOP (\d+)", line)
        if m:
            current_k = int(m.group(1))
        m = row_re.match(line)
        if m and current_k in rows_by_k:
            rows_by_k[current_k].append((int(m.group(1)), float(m.group(2)), float(m.group(3))))
    recall = {1: {}, 10: {}}
    name = system.lower()
    for line in text.splitlines():
        m = re.search(rf"{dataset}-(\d+)-{name}_(\d+)_idx_uint32\.binRecall @ \d+: ([0-9.eE+-]+)", line)
        if m:
            recall[int(m.group(1))][int(m.group(2))] = float(m.group(3))
    out = []
    for k in (1, 10):
        table = rows_by_k[k][-len(recall[k]):] if recall[k] else []
        for ef, qps, pages in table:
            if ef in recall[k]:
                out.append(point(system, k, ef, qps, recall[k][ef], pages, pages_per_query=True))
    return out

def parse_rummy():
    text, rows, current, time_s = read_dataset_log("deep100m", "rummy"), [], None, None
    for line in text.splitlines():
        m = re.search(r"\[REPORT\] TOP (\d+) EF (\d+)", line)
        if m:
            current, time_s = (int(m.group(1)), int(m.group(2))), None
        m = re.search(r"\[REPORT\] Time ([0-9.eE+-]+)", line)
        if m and current:
            time_s = float(m.group(1))
        m = re.search(r"\[REPORT\] RECALL ([0-9.eE+-]+)", line)
        if m and current and time_s is not None:
            rows.append(point("RUMMY", current[0], current[1], N_QUERIES / time_s, float(m.group(1)) / 100.0, None))
            current = None
    return rows

deep_data = (parse_gustann_dataset("deep100m") + parse_bang_dataset("deep100m") +
             parse_table_dataset("deep100m", "DiskANN") + parse_table_dataset("deep100m", "Starling") +
             parse_spann_dataset("deep100m") + parse_rummy())
if not any(row["system"] == "SPANN" for row in deep_data):
    print("ERROR: SPANN has no valid DEEP100M data in this log.")

fig8, axes8 = plt.subplots(2, 2, figsize=(13, 9), sharex="col")
for ax, k in zip(axes8[0], (1, 10)):
    for system in COLORS:
        rows = [r for r in deep_data if r["system"] == system and r["k"] == k and np.isfinite(r["qps"]) and np.isfinite(r["recall"]) ]
        if rows:
            ax.plot([r["recall"] for r in rows], [r["qps"] for r in rows], marker=MARKERS[system], color=COLORS[system], label=system, linewidth=1.5, markersize=5)
    ax.set_yscale("log")
    ax.set_ylim(1e2, 1e6)
    ax.set_xlim(0.8, 1.0)
    ax.grid(True, which="both", alpha=0.25)
    ax.set_ylabel("Throughput (QPS)")
for ax, k in zip(axes8[1], (1, 10)):
    for system in COLORS:
        rows = [r for r in deep_data if r["system"] == system and r["k"] == k and np.isfinite(r["bandwidth"]) and np.isfinite(r["recall"]) ]
        if rows:
            ax.plot([r["recall"] for r in rows], [r["bandwidth"] for r in rows], marker=MARKERS[system], color=COLORS[system], label=system, linewidth=1.5, markersize=5)
    ax.set_xlim(0.8, 1.0)
    ax.set_ylim(*BANDWIDTH_YLIM)
    ax.set_xlabel("Recall@%d" % k)
    ax.set_ylabel("SSD Bandwidth (GiB/s)")
    ax.grid(True, alpha=0.25)
for ax, k in zip(axes8[0], (1, 10)):
    ax.set_xlabel("Recall@%d" % k)
axes8[0, 0].set_title("Recall@1 (k=1)")
axes8[0, 1].set_title("Recall@10 (k=10)")
fig8.suptitle("Figure 8", fontsize=16)
handles8, labels8 = axes8[0, 0].get_legend_handles_labels()
fig8.legend(handles8, labels8, loc="lower center", ncol=6, frameon=False)
fig8.tight_layout(rect=(0, 0.06, 1, 0.96))
output8 = FIGURE_DIR / "figure8_deep100m.pdf"
fig8.savefig(output8, dpi=300, bbox_inches="tight")
print(f"Saved {output8}")
plt.show()


In [ ]:
def read_1b_log(dataset, system):
    path = LOG_DIR / f"log_{dataset}_{system}.log"
    return path.read_text(errors="replace") if path.exists() else ""

def parse_1b_gustann(dataset):
    text = read_1b_log(dataset, "gustann")
    rows, current, time_s, pages = [], None, None, None
    for line in text.splitlines():
        # SIFT1B's main sweep ends before mock, latency, nonav and profiling.
        if re.search(r"\[REPORT\] TOP \d+ EF \d+ MOCK", line):
            break
        m = re.search(r"\[REPORT\] TOP (\d+) EF (\d+)", line)
        if m:
            current, time_s, pages = (int(m.group(1)), int(m.group(2))), None, None
        m = re.search(r"\[REPORT\] Time ([0-9.eE+-]+)", line)
        if m and current:
            time_s = float(m.group(1))
        m = re.search(r"\[REPORT\] IO ([0-9.eE+-]+)", line)
        if m and current:
            pages = float(m.group(1))
        m = re.search(r"\[REPORT\] RECALL: ([0-9.eE+-]+)", line)
        if m and current and time_s is not None and pages is not None:
            qps = N_QUERIES * GUSTANN_REPEAT / time_s
            bandwidth = pages * PAGE_BYTES / time_s / GIB
            rows.append({"system": "GustANN", "k": current[0], "ef": current[1], "qps": qps, "recall": float(m.group(1)), "bandwidth": bandwidth})
            current = None
    return rows

def parse_1b_diskann(dataset):
    text = read_1b_log(dataset, "diskann")
    table, current_k = {1: [], 10: []}, None
    row_re = re.compile(r"^\s*(\d+)\s+4\s+([0-9.eE+-]+)\s+[0-9.eE+-]+\s+[0-9.eE+-]+\s+([0-9.eE+-]+)\s+")
    for line in text.splitlines():
        if re.search(r"\[REPORT\] TOP \d+ MOCK", line):
            current_k = None
            continue
        m = re.search(r"\[REPORT\] TOP (\d+)", line)
        if m:
            current_k = int(m.group(1))
        m = row_re.match(line)
        if m and current_k in table:
            table[current_k].append((int(m.group(1)), float(m.group(2)), float(m.group(3))))
    recall = {1: {}, 10: {}}
    for line in text.splitlines():
        m = re.search(rf"{dataset}-(\d+)-diskann_(\d+)_idx_uint32\.binRecall @ \d+: ([0-9.eE+-]+)", line)
        if m:
            recall[int(m.group(1))][int(m.group(2))] = float(m.group(3))
    rows = []
    for k in (1, 10):
        for ef, qps, pages in table[k][-len(recall[k]):]:
            if ef in recall[k]:
                rows.append({"system": "DiskANN", "k": k, "ef": ef, "qps": qps, "recall": recall[k][ef], "bandwidth": pages * PAGE_BYTES * qps / GIB})
    return rows

def plot_1b_figure(dataset, title, filename):
    rows = parse_1b_gustann(dataset) + parse_1b_diskann(dataset)
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex="col")
    for ax, k in zip(axes[0], (1, 10)):
        for system in ("GustANN", "DiskANN"):
            points = [r for r in rows if r["system"] == system and r["k"] == k]
            if points:
                ax.plot([r["recall"] for r in points], [r["qps"] for r in points], marker=MARKERS[system], color=COLORS[system], label=system, linewidth=1.7, markersize=5)
        ax.set_title("Recall@%d (k=%d)" % (k, k))
        ax.set_xlabel("Recall@%d" % k)
        ax.set_xlim(0.8, 1.0)
        ax.set_ylim(1e3, 1e6)
        ax.set_yscale("log")
        ax.grid(True, which="both", alpha=0.25)
        ax.set_ylabel("Throughput (QPS)")
    for ax, k in zip(axes[1], (1, 10)):
        for system in ("GustANN", "DiskANN"):
            points = [r for r in rows if r["system"] == system and r["k"] == k]
            if points:
                ax.plot([r["recall"] for r in points], [r["bandwidth"] for r in points], marker=MARKERS[system], color=COLORS[system], label=system, linewidth=1.7, markersize=5)
        ax.set_xlabel("Recall@%d" % k)
        ax.set_xlim(0.8, 1.0)
        ax.set_ylim(*BANDWIDTH_YLIM)
        ax.set_ylabel("SSD Bandwidth (GiB/s)")
        ax.grid(True, alpha=0.25)
    fig.suptitle(title, fontsize=16)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, 0.06, 1, 0.94))
    output = FIGURE_DIR / filename
    fig.savefig(output, dpi=300, bbox_inches="tight")
    print(f"Saved {output}")
    plt.show()

plot_1b_figure("sift1b", "Figure 9", "figure9_sift1b.pdf")


In [ ]:
plot_1b_figure("deep1b", "Figure 10", "figure10_deep1b.pdf")


In [ ]:
# Figure 11: SIFT100M, k=10; script repetition counts are 500 and 50.
bam_repeat = 50
pcie_rx_threshold = 1024  # MB/s; excludes loading traffic before search.
systems11 = ("GustANN", "BaM-ANN")
colors11 = {"GustANN": COLORS["GustANN"], "BaM-ANN": "#882255"}
markers11 = {"GustANN": MARKERS["GustANN"], "BaM-ANN": "s"}

def parse_sift_runs11(system, text=None):
    name, repeat = ("gustann", GUSTANN_REPEAT) if system == "GustANN" else ("bam", bam_repeat)
    text = read_log(name) if text is None else text
    # A command header also identifies the unlabelled final PCIe run.
    commands = re.split(r"(?m)^.*\[search_disk\.cu:\d+\] Num ctlrs:", text)[1:]
    rows = []
    for command in commands:
        header = command.splitlines()[0]
        match = re.search(r"topk: (\d+), ef_search: (\d+)", header)
        if not match or int(match.group(1)) != 10:
            continue
        recall = re.findall(r"\[REPORT\] RECALL: ([0-9.eE+-]+)", command)
        elapsed = re.findall(r"\[REPORT\] Time ([0-9.eE+-]+)", command)
        if not elapsed:
            elapsed = re.findall(r"Use time: ([0-9.eE+-]+)", command)
        if not recall or not elapsed:
            continue
        seconds = float(elapsed[-1])
        if seconds <= 0:
            continue
        if system == "GustANN":
            reads = re.findall(r"\[REPORT\] IO ([0-9.eE+-]+)", command)
        else:
            # Actual cache-miss reads, not logical #Accesses (includes hits).
            reads = re.findall(r"#READ IOs: (\d+)", command)
        pages = float(reads[-1]) if reads else math.nan
        query_count = re.findall(r"Read (\d+) queries", command)
        queries = int(query_count[-1]) if query_count else N_QUERIES
        rows.append({"ef": int(match.group(2)), "recall": float(recall[-1]),
                     "qps": queries * repeat / seconds,
                     "ssd": pages * PAGE_BYTES / seconds / GIB,
                     "monitor": "=== PCIe LOG ===" in command, "pcie_log": command})
    return text, rows

def pcie_average11(text):
    return pcie_rx_average(text, threshold=pcie_rx_threshold)

curves11, bars11 = {}, {}
for system in systems11:
    text, runs = parse_sift_runs11(system)
    curves11[system] = [run for run in runs if not run["monitor"]]
    monitored = [run for run in runs if run["monitor"] and run["ef"] == 30]
    if not curves11[system]:
        print(f"ERROR: {system} has no valid SIFT100M k=10 throughput data.")
    try:
        if not monitored or not np.isfinite(monitored[-1]["ssd"]):
            raise ValueError("Missing valid SSD statistics for the ef=30 PCIe run")
        measurement = monitored[-1]
        pcie, samples = pcie_average11(measurement["pcie_log"])
        bars11[system] = (pcie, measurement["ssd"])
        print(f"{system}: PCIe RX {pcie:.3f} GiB/s ({len(samples)} retained seconds), SSD {measurement['ssd']:.3f} GiB/s")
    except ValueError as error:
        print(f"ERROR: {system}: {error}")
        bars11[system] = (math.nan, math.nan)

fig11, axes11 = plt.subplots(1, 2, figsize=(13, 5.5))
for system in systems11:
    rows = curves11[system]
    axes11[0].plot([r["recall"] for r in rows], [r["qps"] for r in rows],
                   color=colors11[system], marker=markers11[system], label=system, linewidth=1.7, markersize=5)
axes11[0].set(xlim=(0.8, 1.0), ylim=(20000, 500000), yscale="log",
              xlabel="Recall@10", ylabel="Throughput (QPS)")
axes11[0].set_yticks([20000, 50000, 100000, 200000, 500000], ["20k", "50k", "100k", "200k", "500k"])
axes11[0].grid(True, which="major", alpha=0.25)
x11, width11 = np.arange(2), 0.32
for index, system in enumerate(systems11):
    bars = axes11[1].bar(x11 + (index - 0.5) * width11, bars11[system], width11,
                         label=system, color=colors11[system], zorder=3)
axes11[1].set_xticks(x11, ["PCIe RX", "SSD"])
axes11[1].set(ylabel="Bandwidth (GiB/s)", ylim=BANDWIDTH_YLIM)
axes11[1].grid(axis="y", alpha=0.25, zorder=0)
fig11.suptitle("Figure 11", fontsize=16)
handles11, labels11 = axes11[0].get_legend_handles_labels()
fig11.legend(handles11, labels11, loc="lower center", ncol=2, frameon=False)
fig11.tight_layout(rect=(0, 0.08, 1, 0.94))
output11 = FIGURE_DIR / "figure11_gustann_bam.pdf"
fig11.savefig(output11, bbox_inches="tight")
print(f"Saved {output11}")
plt.show()


In [ ]:
# SIFT1B k=10, main and NO_NAV sweeps from run_sift1b_gustann.sh.
def parse_pivot_runs12(text):
    headers = list(re.finditer(r"(?m)^\[REPORT\] TOP (\d+) EF (\d+)([^\n]*)$", text))
    rows = {}
    for index, header in enumerate(headers):
        k, ef = int(header.group(1)), int(header.group(2))
        # Only plain throughput sweeps; MOCK and CTX use different workloads.
        if k != 10 or header.group(3).strip():
            continue
        end = headers[index + 1].start() if index + 1 < len(headers) else len(text)
        block = text[header.end():end]
        elapsed = re.findall(r"\[REPORT\] Time ([0-9.eE+-]+)", block)
        reads = re.findall(r"\[REPORT\] IO ([0-9.eE+-]+)", block)
        recall = re.findall(r"\[REPORT\] RECALL: ([0-9.eE+-]+)", block)
        if not elapsed or not reads or not recall:
            continue
        seconds, pages = float(elapsed[-1]), float(reads[-1])
        if seconds <= 0 or not math.isfinite(seconds) or pages < 0:
            continue
        # NAV_GRAPH emits this message; NO_NAV omits it (ssd_search_hybrid.cu).
        variant = "w/ pivot search" if "Use small navigation graph!" in block else "w/o pivot search"
        query_count = re.findall(r"Read (\d+) queries", block)
        queries = int(query_count[-1]) if query_count else N_QUERIES
        # Keep the first sweep per variant/EF, excluding later profiling repeats.
        rows.setdefault((variant, ef), {"variant": variant, "ef": ef,
                        "qps": queries * GUSTANN_REPEAT / seconds,
                        "bandwidth": pages * PAGE_BYTES / seconds / GIB,
                        "recall": float(recall[-1])})
    return list(rows.values())

path12 = LOG_DIR / "log_sift1b_gustann.log"
data12 = parse_pivot_runs12(path12.read_text(errors="replace") if path12.exists() else "")
styles12 = {"w/ pivot search": (COLORS["GustANN"], "o"),
            "w/o pivot search": ("#882255", "s")}
fig12, axes12 = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True)
for variant, (color, marker) in styles12.items():
    rows = sorted([r for r in data12 if r["variant"] == variant], key=lambda r: r["ef"])
    if not rows:
        print(f"ERROR: Figure 12 has no valid SIFT1B k=10 data for {variant}.")
        continue
    for ax, metric in zip(axes12, ("qps", "bandwidth")):
        ax.plot([r["recall"] for r in rows], [r[metric] for r in rows],
                label=variant, color=color, marker=marker, linewidth=1.7, markersize=5)
axes12[0].set(ylim=(0, 500000), yscale="linear", ylabel="Throughput (QPS)")
axes12[0].set_yticks([0, 100000, 200000, 300000, 400000, 500000],
                    ["0", "100k", "200k", "300k", "400k", "500k"])
axes12[1].set(ylim=BANDWIDTH_YLIM, ylabel="SSD Bandwidth (GiB/s)")
for ax in axes12:
    ax.set(xlim=(0.8, 1.0), xlabel="Recall@10")
    ax.grid(True, alpha=0.25)
fig12.suptitle("Figure 12", fontsize=16)
handles12, labels12 = axes12[0].get_legend_handles_labels()
fig12.legend(handles12, labels12, loc="lower center", ncol=2, frameon=False)
fig12.tight_layout(rect=(0, 0.08, 1, 0.94))
output12 = FIGURE_DIR / "figure12_pivot_search.pdf"
fig12.savefig(output12, bbox_inches="tight")
print(f"Saved {output12}")
plt.show()


In [ ]:
probe_k13, probe_ef13 = 10, 30

def parse_ssd_percentiles13(text, k=10, ef=30):
    variants = ("w/o nav graph", "w/ nav graph")
    result = {variant: None for variant in variants}
    commands = re.split(r"(?m)^.*\[search_disk\.cu:\d+\] Num ctlrs:", text)[1:]
    for command in commands:
        header = command.splitlines()[0]
        match = re.search(r"topk: (\d+), ef_search: (\d+)", header)
        if not match or (int(match.group(1)), int(match.group(2))) != (k, ef):
            continue
        # Complete searches only; omit mock queries and CTX latency cases.
        if not re.search(r"\[REPORT\] RECALL: [0-9.eE+-]+", command):
            continue
        query_path = re.search(r"Query File: ([^,]+),", header)
        if query_path and "_arg" in query_path.group(1):
            continue
        if re.search(r"\[REPORT\] TOP \d+ EF \d+ CTX", command):
            continue
        variant = "w/ nav graph" if "Use small navigation graph!" in command else "w/o nav graph"
        percentiles, curves, invalid = None, {}, False
        for line in command.splitlines():
            row = re.fullmatch(r"\s*\[REPORT\] SSD (\d+)[ \t]+([^\r\n]+)\s*", line)
            if row:
                try:
                    seconds = np.array([float(value) for value in row.group(2).split()])
                except ValueError:
                    invalid = True
                    continue
                if (percentiles is None or len(seconds) != len(percentiles) or
                    not np.all(np.isfinite(seconds)) or np.any(seconds < 0) or
                    np.any(np.diff(seconds) < 0)):
                    invalid = True
                    continue
                curves[int(row.group(1))] = (percentiles.copy(), seconds * 1e6)
                continue
            # print_stats emits an unlabelled percentage header before SSD rows.
            try:
                values = np.array([float(value) for value in line.split()])
            except ValueError:
                continue
            if (len(values) >= 2 and np.all(np.isfinite(values)) and
                np.all((values >= 90) & (values <= 100)) and np.all(np.diff(values) > 0)):
                percentiles = values
        expected = {int(value) for value in re.findall(r"Namespace ID:.*\(SSD (\d+)\)", command)}
        if curves and not invalid and (not expected or set(curves) == expected):
            # Prefer the last complete probe run (the NAV profile sweep).
            result[variant] = curves
    return result

path13 = LOG_DIR / "log_sift1b_gustann.log"
data13 = parse_ssd_percentiles13(path13.read_text(errors="replace") if path13.exists() else "", probe_k13, probe_ef13)
missing13 = [variant for variant, curves in data13.items() if not curves]
if missing13:
    print(f"ERROR: Figure 13 missing per-SSD latency percentiles for {', '.join(missing13)} (SIFT1B k={probe_k13}, ef={probe_ef13}); DATA_PROBE must be enabled when collecting logs. PDF not exported.")
else:
    fig13, axes13 = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
    ssds13 = sorted(set().union(*(curves.keys() for curves in data13.values())))
    cmap13 = plt.get_cmap("tab10")
    colors13 = {ssd: cmap13(index % 10) for index, ssd in enumerate(ssds13)}
    for ax, (variant, curves) in zip(axes13, data13.items()):
        for ssd, (percentage, latency_us) in sorted(curves.items()):
            ax.plot(percentage, latency_us, color=colors13[ssd], label=f"SSD {ssd}",
                    marker="o", markersize=3, linewidth=1.5)
        ax.set(title=variant, xlim=(90, 100), xlabel="Percentage (%)", ylabel="Latency (us)")
        ax.set_xticks(np.arange(90, 101, 2))
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False)
    max_latency13 = max(float(np.max(latency)) for curves in data13.values() for _, latency in curves.values())
    axes13[0].set_ylim(0, max_latency13 * 1.05 if max_latency13 > 0 else 1)
    fig13.suptitle("Figure 13", fontsize=16)
    fig13.tight_layout(rect=(0, 0, 1, 0.94))
    output13 = FIGURE_DIR / "figure13_ssd_latency.pdf"
    fig13.savefig(output13, bbox_inches="tight")
    print(f"Saved {output13}")
    plt.show()


In [ ]:
# DEEP100M replication: k=10, ef_search/L=45;
reps14 = np.arange(1, 10)
record_lengths14 = 96 * 4 * reps14 + 512

def parse_rep_runs14(text, system):
    headers = list(re.finditer(r"(?m)^\[REPORT\] Rep (\d+)[ \t]*$", text))
    rows = {}
    for index, header in enumerate(headers):
        rep = int(header.group(1))
        end = headers[index + 1].start() if index + 1 < len(headers) else len(text)
        block = text[header.end():end]
        if system == "GustANN":
            elapsed = re.findall(r"\[REPORT\] Time ([0-9.eE+-]+)", block)
            reads = re.findall(r"\[REPORT\] IO ([0-9.eE+-]+)", block)
            complete = re.search(r"\[REPORT\] RECALL: ([0-9.eE+-]+)", block)
            if not elapsed or not reads or not complete:
                continue
            seconds, pages = float(elapsed[-1]), float(reads[-1])
            if not math.isfinite(seconds) or seconds <= 0 or not math.isfinite(pages) or pages < 0:
                continue
            counts = re.findall(r"Read (\d+) queries", block)
            queries = int(counts[-1]) if counts else N_QUERIES
            qps = queries * GUSTANN_REPEAT / seconds
            ssd = pages * PAGE_BYTES / seconds / GIB
            try:
                pcie, samples = pcie_rx_average(block)
                retained = len(samples)
            except ValueError:
                pcie, retained = math.nan, 0
            rows[rep] = {"qps": qps, "ssd": ssd, "pcie": pcie,
                         "effective": ssd * (96 * 4 * rep + 512) / PAGE_BYTES,
                         "pcie_samples": retained}
        else:
            table = []
            reading_table = False
            for line in block.splitlines():
                if "QPS" in line and "Mean IOs" in line:
                    reading_table = True
                    continue
                if not reading_table:
                    continue
                match = re.fullmatch(r"\s*45\s+\d+\s+([0-9.eE+-]+)\s+[0-9.eE+-]+\s+[0-9.eE+-]+\s+([0-9.eE+-]+)\s+[0-9.eE+-]+\s+[0-9.eE+-]+\s*", line)
                if match:
                    table.append((float(match.group(1)), float(match.group(2))))
                elif table and line.strip() and not line.startswith("="):
                    reading_table = False
            if table:
                # As in the earlier figures, take the final repeated L run.
                qps, pages_per_query = table[-1]
                if math.isfinite(qps) and qps > 0 and math.isfinite(pages_per_query) and pages_per_query >= 0:
                    rows[rep] = {"qps": qps, "ssd": qps * pages_per_query * PAGE_BYTES / GIB}
    return rows

data14, errors14 = {}, []
for system in ("GustANN", "DiskANN"):
    path = LOG_DIR / f"log_rep_{system.lower()}.log"
    data14[system] = parse_rep_runs14(path.read_text(errors="replace") if path.exists() else "", system)
    missing = [str(rep) for rep in reps14 if rep not in data14[system]]
    if missing:
        errors14.append(f"{system} missing completed runs for rep={','.join(missing)}")
missing_pcie14 = [str(rep) for rep in reps14 if rep in data14["GustANN"] and not np.isfinite(data14["GustANN"][rep]["pcie"])]
if missing_pcie14:
    errors14.append(f"GustANN missing valid PCIe samples for rep={','.join(missing_pcie14)}")
if errors14:
    print("ERROR: Figure 14: " + "; ".join(errors14) + ". Missing values are left blank.")

fig14, qps_axis14 = plt.subplots(figsize=(12, 6))
bw_axis14 = qps_axis14.twinx()
specs14 = [
    ("GustANN", "qps", "GustANN Throughput", qps_axis14, COLORS["GustANN"], "o", "-"),
    ("GustANN", "ssd", "GustANN SSD B/W", bw_axis14, "#D55E00", "s", "--"),
    ("GustANN", "pcie", "GustANN PCIe B/W", bw_axis14, "#882255", "D", "--"),
    ("GustANN", "effective", "GustANN Effective B/W", bw_axis14, "#E69F00", "P", "--"),
    ("DiskANN", "qps", "DiskANN Throughput", qps_axis14, COLORS["DiskANN"], "^", "-"),
    ("DiskANN", "ssd", "DiskANN SSD B/W", bw_axis14, "#555555", "v", "--"),
]
lines14 = []
for system, metric, label, axis, color, marker, linestyle in specs14:
    values = [data14[system].get(int(rep), {}).get(metric, math.nan) for rep in reps14]
    line, = axis.plot(record_lengths14, values, label=label, color=color, marker=marker,
                      linestyle=linestyle, linewidth=1.7, markersize=5)
    lines14.append(line)
qps_axis14.set(xlabel="Record Length (bytes)", ylabel="Throughput (QPS)", ylim=(0, 300000), yscale="linear")
qps_axis14.set_yticks([0, 50000, 100000, 150000, 200000, 250000, 300000],
                     ["0", "50k", "100k", "150k", "200k", "250k", "300k"])
qps_axis14.set_xticks([1024, 2048, 3072, 4096], ["1K", "2K", "3K", "4K"])
qps_axis14.grid(True, alpha=0.25)
# GB/s uses 1024**3 bytes/s, consistent with all preceding bandwidth plots.
bw_axis14.set(ylabel="Bandwidth (GB/s)", ylim=(0, 50))
fig14.suptitle("Figure 14", fontsize=16)
fig14.legend(lines14, [line.get_label() for line in lines14],
             loc="lower center", ncol=3, frameon=False)
fig14.tight_layout(rect=(0, 0.13, 1, 0.94))
output14 = FIGURE_DIR / "figure14_rep.pdf"
fig14.savefig(output14, bbox_inches="tight")
print(f"Saved {output14}")
plt.show()


In [ ]:
def parse_ssd_scaling15(text, system, count):
    headers = list(re.finditer(r"(?m)^\[REPORT\] SIFT100M (GustANN|DiskANN) SSD (\d+) TOP (\d+) (EF|L) (\d+)[ \t]*$", text))
    if not headers:
        raise ValueError("scaling experiment header is missing")
    header = headers[-1]
    expected_mode = "EF" if system == "GustANN" else "L"
    if (header.group(1), int(header.group(2)), int(header.group(3)), header.group(4), int(header.group(5))) != (system, count, 10, expected_mode, 30):
        raise ValueError("header does not match the filename or k=10, ef_search/L=30")
    block = text[header.end():]
    if system == "GustANN":
        times = re.findall(r"\[REPORT\] Time ([0-9.eE+-]+)", block)
        reads = re.findall(r"\[REPORT\] IO ([0-9.eE+-]+)", block)
        if not times or not reads or not re.search(r"\[REPORT\] RECALL: [0-9.eE+-]+", block):
            raise ValueError("completed search statistics are missing")
        seconds, pages = float(times[-1]), float(reads[-1])
        if not math.isfinite(seconds) or seconds <= 0 or not math.isfinite(pages) or pages < 0:
            raise ValueError("invalid runtime or SSD page count")
        counts = re.findall(r"Read (\d+) queries", block)
        queries = int(counts[-1]) if counts else N_QUERIES
        qps = queries * GUSTANN_REPEAT / seconds
        bandwidth = pages * PAGE_BYTES / seconds / GIB
    else:
        table = re.search(r"(?m)^.*QPS.*Mean IOs.*$", block)
        if not table or not re.search(r"Recall @ 10: [0-9.eE+-]+", block):
            raise ValueError("completed search table or recall is missing")
        rows = re.findall(r"(?m)^[ \t]*30[ \t]+\d+[ \t]+([0-9.eE+-]+)[ \t]+[0-9.eE+-]+[ \t]+[0-9.eE+-]+[ \t]+([0-9.eE+-]+)[ \t]+[0-9.eE+-]+[ \t]+[0-9.eE+-]+[ \t]*$", block[table.end():])
        if not rows:
            raise ValueError("L=30 search statistics are missing")
        qps, pages_per_query = map(float, rows[-1])
        if not math.isfinite(pages_per_query) or pages_per_query < 0:
            raise ValueError("invalid mean SSD page count")
        bandwidth = qps * pages_per_query * PAGE_BYTES / GIB
    if not math.isfinite(qps) or qps <= 0 or not math.isfinite(bandwidth):
        raise ValueError("invalid throughput or SSD bandwidth")
    return {"qps": qps, "bandwidth": bandwidth}

def load_ssd_scaling15(directory):
    files, data, errors = {}, {"GustANN": {}, "DiskANN": {}}, []
    for path in directory.glob("log_ssd_*.log"):
        match = re.fullmatch(r"log_ssd_(gustann|diskann)_([1-9][0-9]*)\.log", path.name)
        if match:
            system = "GustANN" if match.group(1) == "gustann" else "DiskANN"
            files[(system, int(match.group(2)))] = path
    counts = sorted({count for _, count in files})
    for system in data:
        for count in counts:
            path = files.get((system, count))
            if path is None:
                errors.append(f"{system} SSD {count}: log is missing")
                continue
            try:
                data[system][count] = parse_ssd_scaling15(path.read_text(errors="replace"), system, count)
            except (ValueError, OSError) as error:
                errors.append(f"{system} SSD {count}: {error}")
    return counts, data, errors

ssd_counts15, data15, errors15 = load_ssd_scaling15(LOG_DIR)
if errors15:
    print("ERROR: Figure 15: " + "; ".join(errors15) + ". Missing values are left blank.")
if not ssd_counts15 or not any(data15.values()):
    print("ERROR: Figure 15 has no valid SSD scaling results in this log directory. PDF not exported.")
else:
    fig15, axes15 = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True)
    positions15, width15 = np.arange(len(ssd_counts15)), 0.36
    for index, system in enumerate(("GustANN", "DiskANN")):
        for ax, metric in zip(axes15, ("qps", "bandwidth")):
            values = [data15[system].get(count, {}).get(metric, math.nan) for count in ssd_counts15]
            ax.bar(positions15 + (index - 0.5) * width15, values, width15,
                   label=system, color=COLORS[system], zorder=3)
    axes15[0].set(yscale="linear", ylim=(0, 400000), ylabel="Throughput (QPS)")
    axes15[0].set_yticks([0, 100000, 200000, 300000, 400000], ["0", "100k", "200k", "300k", "400k"])
    axes15[1].set(ylim=BANDWIDTH_YLIM, ylabel="SSD Bandwidth (GiB/s)")
    for ax in axes15:
        ax.set_xlabel("Number of SSDs")
        ax.set_xticks(positions15, [str(count) for count in ssd_counts15])
        ax.grid(axis="y", alpha=0.25, zorder=0)
    fig15.suptitle("Figure 15", fontsize=16)
    handles15, labels15 = axes15[0].get_legend_handles_labels()
    fig15.legend(handles15, labels15, loc="lower center", ncol=2, frameon=False)
    fig15.tight_layout(rect=(0, 0.08, 1, 0.94))
    output15 = FIGURE_DIR / "figure15_ssd_scaling.pdf"
    fig15.savefig(output15, bbox_inches="tight")
    print(f"Saved {output15}")
    plt.show()


In [ ]:
mock_repeat16 = 5

def parse_mock16(text, system):
    headers = list(re.finditer(r"(?m)^\[REPORT\] TOP (\d+)([^\n]*)$", text))
    rows, errors = {}, []
    found = False
    for index, header in enumerate(headers):
        if int(header.group(1)) != 10:
            continue
        mode = header.group(2).strip()
        if system == "GustANN":
            match = re.fullmatch(r"EF (\d+) MOCK", mode)
            if not match:
                continue
            length = int(match.group(1))
        elif mode != "MOCK":
            continue
        found = True
        end = headers[index + 1].start() if index + 1 < len(headers) else len(text)
        block = text[header.end():end]
        if system == "GustANN":
            times = re.findall(r"\[REPORT\] Time ([0-9.eE+-]+)", block)
            reads = re.findall(r"\[REPORT\] IO ([0-9.eE+-]+)", block)
            counts = re.findall(r"Read (\d+) queries", block)
            if not times or not reads or not counts:
                errors.append(f"{system} L={length}: runtime, IO or query count is missing")
                continue
            seconds, pages, queries = float(times[-1]), float(reads[-1]), int(counts[-1])
            if seconds <= 0 or not math.isfinite(seconds) or pages < 0 or not math.isfinite(pages) or queries <= 0:
                errors.append(f"{system} L={length}: invalid search statistics")
                continue
            rows[length] = {"qps": queries * mock_repeat16 / seconds,
                            "bandwidth": pages * PAGE_BYTES / seconds / GIB}
        else:
            table = re.search(r"(?m)^.*QPS.*Mean IOs.*$", block)
            if not table:
                errors.append("DiskANN: MOCK search table is missing")
                continue
            # Only search table rows; no ground truth exists for MOCK queries.
            table_text = block[table.end():].split("Done searching", 1)[0]
            values = re.findall(r"(?m)^[ \t]*(\d+)[ \t]+\d+[ \t]+([0-9.eE+-]+)[ \t]+[0-9.eE+-]+[ \t]+[0-9.eE+-]+[ \t]+([0-9.eE+-]+)[ \t]+[0-9.eE+-]+[ \t]+[0-9.eE+-]+[ \t]*$", table_text)
            if not values:
                errors.append("DiskANN: MOCK search statistics are missing")
            for length, qps, pages in values:
                length, qps, pages = int(length), float(qps), float(pages)
                if qps <= 0 or not math.isfinite(qps) or pages < 0 or not math.isfinite(pages):
                    errors.append(f"DiskANN L={length}: invalid search statistics")
                    continue
                # Later repetitions overwrite earlier ones for the same L.
                rows[length] = {"qps": qps, "bandwidth": qps * pages * PAGE_BYTES / GIB}
    if not found:
        errors.append(f"{system}: SIFT1B k=10 MOCK section is missing")
    return rows, errors

data16, errors16 = {}, []
for system in ("GustANN", "DiskANN"):
    path = LOG_DIR / f"log_sift1b_{system.lower()}.log"
    data16[system], errors = parse_mock16(path.read_text(errors="replace") if path.exists() else "", system)
    errors16.extend(errors)
lengths16 = sorted(set().union(*(rows.keys() for rows in data16.values())))
for system, rows in data16.items():
    missing = [str(length) for length in lengths16 if length not in rows]
    if missing:
        errors16.append(f"{system}: missing L={','.join(missing)}")
if errors16:
    print("ERROR: Figure 16: " + "; ".join(errors16) + ". Missing values are left blank.")
if not lengths16:
    print("ERROR: Figure 16 has no valid SIFT1B MOCK data. PDF not exported.")
else:
    fig16, axes16 = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True)
    for system, rows in data16.items():
        for ax, metric in zip(axes16, ("qps", "bandwidth")):
            values = [rows.get(length, {}).get(metric, math.nan) for length in lengths16]
            ax.plot(lengths16, values, label=system, color=COLORS[system],
                    marker=MARKERS[system], linewidth=1.7, markersize=5)
    axes16[0].set(yscale="log", ylim=(1000, 500000), ylabel="Throughput (QPS)")
    axes16[1].set(ylim=BANDWIDTH_YLIM, ylabel="SSD Bandwidth (GiB/s)")
    for ax in axes16:
        ax.set_xlabel("search length")
        ax.set_xticks(lengths16)
        ax.grid(True, which="major", alpha=0.25)
    fig16.suptitle("Figure 16", fontsize=16)
    handles16, labels16 = axes16[0].get_legend_handles_labels()
    fig16.legend(handles16, labels16, loc="lower center", ncol=2, frameon=False)
    fig16.tight_layout(rect=(0, 0.08, 1, 0.94))
    output16 = FIGURE_DIR / "figure16_sift1b_mock.pdf"
    fig16.savefig(output16, bbox_inches="tight")
    print(f"Saved {output16}")
    plt.show()


In [ ]:
def parse_ctx17(text):
    headers = list(re.finditer(r"(?m)^\[REPORT\] TOP (\d+) EF (\d+)([^\n]*)$", text))
    rows, errors = {}, []
    for index, header in enumerate(headers):
        match = re.fullmatch(r"CTX ([0-9]+(?:x[0-9]+){1,2})", header.group(3).strip())
        if int(header.group(1)) != 10 or not match:
            continue
        label = match.group(1)
        dimensions = tuple(map(int, label.split("x")))
        # The CxT sweep uses B=1120 in run_sift1b_gustann.sh.
        batch, contexts, threads = dimensions if len(dimensions) == 3 else (1120, *dimensions)
        if min(batch, contexts, threads) <= 0:
            errors.append(f"CTX {label}: invalid B/C/T dimensions")
            continue
        end = headers[index + 1].start() if index + 1 < len(headers) else len(text)
        block = text[header.end():end]
        times = re.findall(r"\[REPORT\] Time ([0-9.eE+-]+)", block)
        latency = {int(t): float(value) for t, value in re.findall(r"\[REPORT\] LAT(\d+) ([0-9.eE+-]+)", block)}
        query_counts = {int(t): int(count) for t, count in re.findall(r"Thread (\d+):[^\n]*?\s(\d+) queries in", block)}
        expected = set(range(threads))
        if not times or set(latency) != expected or set(query_counts) != expected:
            errors.append(f"CTX {label}: runtime, per-thread latency or query counts are missing")
            continue
        seconds = float(times[-1])
        if (not math.isfinite(seconds) or seconds <= 0 or
            any(not math.isfinite(value) or value < 0 for value in latency.values()) or
            any(value <= 0 for value in query_counts.values())):
            errors.append(f"CTX {label}: invalid runtime, latency or query counts")
            continue
        # Each batch has B queries, except the single final partial batch.
        batch_counts = [(query_counts[t] + batch - 1) // batch for t in range(threads)]
        latency_ms = float(np.average([latency[t] for t in range(threads)], weights=batch_counts))
        actual_ef = re.findall(r"topk: 10, ef_search: (\d+)", block)
        ef = int(actual_ef[0]) if actual_ef else int(header.group(2))
        rows[(batch, contexts, threads, ef)] = {"ctx": label, "batch": batch,
            "contexts": contexts, "threads": threads, "ef": ef,
            "latency_ms": latency_ms, "qps": sum(query_counts.values()) / seconds}
    return sorted(rows.values(), key=lambda row: row["latency_ms"]), errors

path17 = LOG_DIR / "log_sift1b_gustann.log"
data17, errors17 = parse_ctx17(path17.read_text(errors="replace") if path17.exists() else "")
if errors17:
    print("ERROR: Figure 17: " + "; ".join(errors17) + ". Incomplete cases are omitted.")
if not data17:
    print("ERROR: Figure 17 has no valid SIFT1B CTX data. PDF not exported.")
else:
    fig17, axis17 = plt.subplots(figsize=(8, 5.5))
    axis17.plot([row["latency_ms"] for row in data17], [row["qps"] for row in data17],
                label="GustANN", color=COLORS["GustANN"], marker=MARKERS["GustANN"],
                linewidth=1.7, markersize=5)
    axis17.set(xlabel="Search Latency (ms)", ylabel="Throughput (QPS)",
               xlim=(0, 250), ylim=(0, 300000), xscale="linear", yscale="linear")
    axis17.set_yticks([0, 50000, 100000, 150000, 200000, 250000, 300000],
                     ["0", "50k", "100k", "150k", "200k", "250k", "300k"])
    axis17.grid(True, alpha=0.25)
    axis17.legend(frameon=False)
    fig17.suptitle("Figure 17", fontsize=16)
    fig17.tight_layout(rect=(0, 0, 1, 0.94))
    output17 = FIGURE_DIR / "figure17_ctx_latency_throughput.pdf"
    fig17.savefig(output17, bbox_inches="tight")
    print(f"Saved {output17}")
    plt.show()
